# Telco Customer Churn  
## Data Preprocessing & Database Loading Pipeline 

This notebook performs data cleaning, validation,
and loads the processed dataset into SQL Server.

importing libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

## 1. Data Loading
Load the raw dataset from Excel file.

In [3]:
df=pd.read_excel('Telco_customer_churn.xlsx')

In [4]:
df

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,2569-WGERO,1,United States,California,Landers,92285,"34.341737, -116.539416",34.341737,-116.539416,Female,...,Two year,Yes,Bank transfer (automatic),21.15,1419.4,No,0,45,5306,NaN
7039,6840-RESVB,1,United States,California,Adelanto,92301,"34.667815, -117.536183",34.667815,-117.536183,Male,...,One year,Yes,Mailed check,84.80,1990.5,No,0,59,2140,NaN
7040,2234-XADUH,1,United States,California,Amboy,92304,"34.559882, -115.637164",34.559882,-115.637164,Female,...,One year,Yes,Credit card (automatic),103.20,7362.9,No,0,71,5560,NaN
7041,4801-JZAZL,1,United States,California,Angelus Oaks,92305,"34.1678, -116.86433",34.167800,-116.864330,Female,...,Month-to-month,Yes,Electronic check,29.60,346.45,No,0,59,2793,NaN


## 2. Initial Data Exploration
Quick structural inspection of dataset
(shape, types, missing values, uniqueness, summary stats).

In [5]:
def explore_data(df):
    print("="*50)
    print("📌 DATA SHAPE")
    print("="*50)
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    
    print("\n" + "="*50)
    print("📌 DATA TYPES")
    print("="*50)
    print(df.dtypes)
    
    print("\n" + "="*50)
    print("📌 MISSING VALUES")
    print("="*50)
    print(df.isnull().sum())
    
    print("\n" + "="*50)
    print("📌 UNIQUE VALUES COUNT")
    print("="*50)
    print(df.nunique())
    
    print("\n" + "="*50)
    print("📌 NUMERICAL SUMMARY")
    print("="*50)
    print(df.describe())
    

In [6]:
explore_data(df)

📌 DATA SHAPE
Rows: 7043
Columns: 33

📌 DATA TYPES
CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          

## 3. Data Cleaning
Removing irrelevant columns and correcting data types.

In [7]:
df.drop(columns=[
    'Count','Zip Code','Longitude','Latitude','Lat Long',
    'Churn Value','Country','State'
], inplace=True)

### Type Correction
Convert `Total Charges` to numeric to ensure proper aggregation and storage.

In [8]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

In [9]:
explore_data(df)

📌 DATA SHAPE
Rows: 7043
Columns: 25

📌 DATA TYPES
CustomerID            object
City                  object
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges        float64
Churn Label           object
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

📌 MISSING VALUES
CustomerID              0
City                    0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Servi

### Handling Missing Values
Remove records where `Total Charges` is missing.

In [10]:
df = df.dropna(subset=['Total Charges'])

In [11]:
explore_data(df)

📌 DATA SHAPE
Rows: 7032
Columns: 25

📌 DATA TYPES
CustomerID            object
City                  object
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges        float64
Churn Label           object
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

📌 MISSING VALUES
CustomerID              0
City                    0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Servi

## 4. Database Connection & Data Insertion
Establish SQL Server connection and load cleaned data.

In [12]:
server = 'localhost'   
database = 'churn_analysis_project'

engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

df.to_sql("telco_churn_data", engine, if_exists="replace", index=False)

C:\Users\moham\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1050.2'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


60

In [13]:
df

,CustomerID,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,Los Angeles,Male,No,No,No,2,Yes,No,DSL,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,86,3239,Competitor made better offer
1,9237-HQITU,Los Angeles,Female,No,No,Yes,2,Yes,No,Fiber optic,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,67,2701,Moved
2,9305-CDSKC,Los Angeles,Female,No,No,Yes,8,Yes,Yes,Fiber optic,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,86,5372,Moved
3,7892-POOKP,Los Angeles,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,...,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,84,5003,Moved
4,0280-XJGEX,Los Angeles,Male,No,No,Yes,49,Yes,Yes,Fiber optic,...,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,89,5340,Competitor had better devices
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,2569-WGERO,Landers,Female,No,No,No,72,Yes,No,No,...,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.40,No,45,5306,NaN
7039,6840-RESVB,Adelanto,Male,No,Yes,Yes,24,Yes,Yes,DSL,...,Yes,One year,Yes,Mailed check,84.80,1990.50,No,59,2140,NaN
7040,2234-XADUH,Amboy,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,...,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No,71,5560,NaN
7041,4801-JZAZL,Angelus Oaks,Female,No,Yes,Yes,11,No,No phone service,DSL,...,No,Month-to-month,Yes,Electronic check,29.60,346.45,No,59,2793,NaN
